In [3]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels as sm
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols, mixedlm
import time
from datetime import date

# updated src file changes are loaded
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath(".."))

In [9]:
# load and combine experimet df

from src.loading import load_data

# -------------------- 3.0 m/s
df_d3 = load_data("../data/target_trial_3_2025_11_7.parquet",  keep_col='training_status')
# ------------------- 2.0 m/s
df_d2 = load_data("../data/target_trial_2.parquet")
# # drop every row from participant p030 due to corrupted data
df_d2 = df_d2[df_d2['ppid'] != 'p030'].copy()

# ------------------- Spatial Generalization 3.0 m/s
df_s3 = load_data("../data/spatial_generalization/spatial_generalization_3m.parquet", keep_col='training_status')
# # drop every row from participant p007
df_s3 = df_s3[df_s3['ppid'] != 'p007'].copy()

# ------------------- Spatial Generalization 2.0 m/s
df_s2 = load_data("../data/target_trial_spatial_generalization.parquet", keep_col='training_status')

df_full = pd.concat([df_d2, df_d3, df_s2, df_s3], ignore_index=True)
df_full

,experiment,ppid,session_num,trial_num,block_num,trial_num_in_block,start_time,end_time,hand,target_hit,...,launch_dev_z,launch_Speed_z,set_order,sign_label,side,target_hit_binary,trial_num_target,target_x_label,training_status,sign
0,projectile_experiment,p000,1,1,1,1,31.91363,71.80201,r,False,...,0.456195,-1.925087,1,pos,Upstream,0.0,1,p0.3,NaN,NaN
1,projectile_experiment,p000,1,2,1,2,71.80201,81.57587,r,False,...,0.804504,-2.255711,1,pos,Upstream,0.0,1,p0.6,NaN,NaN
2,projectile_experiment,p000,1,3,1,3,81.57587,95.72057,r,False,...,0.789440,-2.555764,1,neg,Downstream,0.0,1,neg0.6,NaN,NaN
3,projectile_experiment,p000,1,4,1,4,95.72057,100.81590,r,True,...,1.090968,-1.605964,1,neg,Downstream,1.0,1,neg0.3,NaN,NaN
4,projectile_experiment,p000,1,5,2,1,100.81590,107.42900,r,False,...,0.882501,-2.305389,1,neg,Downstream,0.0,2,neg0.3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55878,novel_TD_3.0M_spatial_generalization,p058,1,385,85,4,1734.46600,1737.15400,r,False,...,1.304581,0.905291,1,pos,Upstream,0.0,84,p0.6,NaN,NaN
55879,novel_TD_3.0M_spatial_generalization,p058,1,386,86,1,1737.15400,1739.99300,r,False,...,1.258949,0.173611,1,pos,Upstream,0.0,21,p0.3,NaN,NaN
55880,novel_TD_3.0M_spatial_generalization,p058,1,387,86,2,1739.99300,1742.83300,r,True,...,0.933559,0.276779,1,neg,Downstream,1.0,262,neg0.6,NaN,NaN
55881,novel_TD_3.0M_spatial_generalization,p058,1,388,86,3,1742.83300,1745.85500,r,True,...,0.997681,-0.356422,1,neg,Downstream,1.0,21,neg0.3,NaN,NaN


In [11]:

# fix water_speed_binary if needed
from pandas.api.types import is_integer_dtype

if not is_integer_dtype(df_full['water_speed_binary']):

    print('water_speed_binary is acting strange: Implementing fix...')
    
    df_full['water_speed_binary2'] = (df_full['water_speed_m_s'] != 0.0).astype(int).copy()
    df_full['water_speed_binary'] = df_full['water_speed_binary2'].copy()
    
    # drops redundant col
    df_full.drop(columns=['water_speed_binary2'], inplace=True)
    


# add new variables

df_full['lateral_error_x'] = (df_full['min_pos_from_target_x'] - df_full['target_position_x']) # 
df_full['depth_error_z'] = (df_full['min_pos_from_target_z'] - df_full['target_position_z']) 

# convert key columns to cm:
from src.convert_to_cm import convert_to_cm

df_labeled = convert_to_cm(df_full,
              cols=[
                    'min_pos_from_target_x',
                    'min_pos_from_target_z',
                    'target_position_x',
                    'target_position_z',
                    'lateral_error_x',
                    'depth_error_z'
                   ])

print('res:', df_labeled['target_position_x'].dtypes)

# label ppid by water speed
from src.label_water_speed_ppid import label_water_condtion_ppid
df_labeled = label_water_condtion_ppid(df_labeled, ppid_col='ppid', water_col='water_speed_m_s', selected_speeds=[-3.0,-2.0,0.0]).copy() 
# one-dimensional errors - in cm
#df_labeled['lateral_error_x_cm'] = (df_labeled['min_pos_from_target_x'] - df_labeled['target_position_x']) * 100 # convert to cm
#df_labeled['depth_error_z_cm'] = (df_labeled['min_pos_from_target_z'] - df_labeled['target_position_z']) * 100

# order targets

# adjust target label
target_mapping = {
    "p0.6": "R60",
    "p0.3": "R30",
    "neg0.3": "L30",
    "neg0.6": "L60"
}
df_labeled['target_x_label'] = df_labeled['target_x_label'].map(target_mapping).copy()

from src.features import order_targets
df_labeled['target_position_x'] = order_targets(df_labeled.copy())
df_labeled['target_x_label'] = order_targets(df_labeled.copy())

print(df_labeled.columns.to_list())
df_labeled['signed_euclidean_cm'] = df_labeled['ball_dist_to_center_cm'] * np.sign(df_labeled['lateral_error_x_cm'])



df_labeled['set_order'] = df_labeled['set_order'].replace({
    '1': '63_36',
    '2': '36_63'
}).copy()

# ensures PCA compatibility
df_labeled['water_speed_m_s'] = df_labeled['water_speed_m_s'].replace(0.0, 0.0001).copy()


# add temporal phase label categorical varible
from src.add_phases import label_phases, df_add_phases
phases = label_phases() 

df_labeled = df_add_phases(df_labeled, phase_array=phases).copy()


from src.add_cycles import add_cycles
df_labeled = add_cycles(df_labeled,
                       n_trials=4,
                       ppid_col='ppid_full',
                       target_col='target_x_label',
                       ).copy()

# filter out familiarization (i.e, starts with an 'f')
df_nf = df_labeled[~df_labeled['phase'].str.startswith('f')].copy()

# remove trials that do not cross a certain threshold in the z-plane (i.e, trials that did not exceed 70 cm in the z-plane)
from src.cleaning import crossed_threshold

# make column checking if ball crossed 70.0 cm on z axis
df_nf['crossed_threshold'] = df_nf.apply(
    lambda row: crossed_threshold(row, col="ball_pos_z", val=0.70), # 70 cm
        axis=1
).copy()
# check
print(df_nf['crossed_threshold'].value_counts())
print(df_nf['ball_pos_z'].describe())

# Make a seperate df without trials that did not cross threshold
df_filt = df_nf[df_nf['crossed_threshold']].copy() # retains rows where crossed_threshold == True

# reduce df to key cols
from src.cleaning import extract_key_columns
df_small_filt = extract_key_columns(df_filt).copy()


# check participant counts per group:
from src.features import get_sample_sizes
get_sample_sizes(df_small_filt, group_cols=['speed_label','experiment', 'set_order'], ppid_col='ppid_full')



water_speed_binary is acting strange: Implementing fix...
res: float64
0        R30
1        R60
2        L60
3        L30
4        L30
        ... 
55878    R60
55879    R30
55880    L60
55881    L30
55882    R60
Name: target_x_label, Length: 55883, dtype: category
Categories (4, object): ['L60' < 'L30' < 'R30' < 'R60']
0        R30
1        R60
2        L60
3        L30
4        L30
        ... 
55878    R60
55879    R30
55880    L60
55881    L30
55882    R60
Name: target_x_label, Length: 55883, dtype: category
Categories (4, object): ['L60' < 'L30' < 'R30' < 'R60']
['experiment', 'ppid', 'session_num', 'trial_num', 'block_num', 'trial_num_in_block', 'start_time', 'end_time', 'hand', 'target_hit', 'final_ball_state', 'type', 'target_position_x', 'target_position_y', 'target_position_z', 'target_width', 'launch_direction', 'before_launch_velocity_x', 'before_launch_velocity_z', 'before_launch_speed', 'before_launch_angle', 'current_force', 'water_inertia', 'water_speed_m_s', 'launch_a

C:\Users\jacob\AppData\Local\Temp\ipykernel_28416\738862041.py:63: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_labeled['set_order'] = df_labeled['set_order'].replace({
C:\Users\jacob\water_current_MA\src\add_phases.py:56: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['phase_target_trial_num'] = df.groupby(['ppid_full', 'phase', 'target_x_label']).cumcount() + 1
C:\Users\jacob\water_current_MA\src\add_cycles.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or

crossed_threshold
True     52820
False      631
Name: count, dtype: int64
count                                                 53451
unique                                                53451
top       0.000000_0.013353_0.026616_0.052881_0.065885_0...
freq                                                      1
Name: ball_pos_z, dtype: object


C:\Users\jacob\water_current_MA\src\features.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_cols).size()


speed_label  experiment                            set_order
-3.0         S1-projectile_experiment              63_36         0
                                                   36_63         0
             S1_3.0M-projectile_experiment 1       63_36        20
                                                   36_63         0
             S2-projectile_experiment              63_36         0
                                                   36_63         0
             S2_3.0M-projectile_experiment 1       63_36         0
                                                   36_63        25
             TD_2.0M_spatial_generalization        63_36         0
                                                   36_63         0
             TU_2.0M_spatial_generalization        63_36         0
                                                   36_63         0
             novel_TD_3.0M_spatial_generalization  63_36         9
                                                   36_63         0
 